In [1]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [2]:
# dp = '/data2/hratch/human_me/'
# preprocess.create_environment(build_path = dp + 'build/', input_path = dp + 'inputs/',
#                   outdir = dp + 'processed/', n_cores=20)

In [3]:
from preprocess import correct_inputs as ci
from utils.load_environmental_variables import *

full model

In [4]:
# prebuild = '/data2/hratch/human_me/prebuild/'
# ci.correct_model(model = prebuild + 'recon2_2.xml')
# revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                             non_machinery = None)
# print(revised_genes)
# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False)
# me_model.pickle('/data2/hratch/human_me/full_12_29_20.pickle')

toy model

In [5]:
# other = '/data2/hratch/human_me/other/'
# ci.correct_model(model = other + 'toy_model.xml')
# revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                                non_machinery = None)
# print(revised_genes)
# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                             unmodeled_protein_frac = None)
# toy_me_model.pickle(other + 'toy_me_12_29_20.pickle')
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

In [6]:
from expression import build_me_model
toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                            unmodeled_protein_frac = None)
sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


Generate ubiquitin reactions for proteasomal degradation
Generate ribosome


  1%|          | 7/591 [00:00<00:09, 60.39it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:11<00:00, 52.48it/s]


...

Check reaction mass balances


 12%|█▏        | 1453/12606 [00:00<00:00, 14503.15it/s]

Check correct coupling of metabolic machinery


100%|██████████| 12606/12606 [00:05<00:00, 2128.65it/s]
../scripts/core/model.py:301 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Time to build: 4.47626705567042 minutes
Getting MINOS parameters...
Done in 257.625 seconds with status 0


# Check

In [11]:
# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                                 unmodeled_protein_frac = None,
#                                                 model_id = 'toy_me_model')
# jabba = True
# if jabba:
#     for r in toy_me_model.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
            
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

In [7]:
# toy_me_model.add_boundary(metabolite = toy_me_model.metabolites.get_by_id('h_c'), 
#                           type = 'demand')
# sln, stat, _ = toy_me_model.solve_lp(mu_val =  1e-9)

../scripts/core/model.py:259 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 281.157 seconds with status 0


In [15]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in toy_me_model.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln[toy_me_model.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,4.921013e-10
protein_biomass_to_biomass,protein_biomass_to_biomass,3.258721e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
DNA_biomass_to_biomass,DNA_biomass_to_biomass,1.400000e-11


In [16]:
S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
fn = '/data2/hratch/human_me/other/test_lp/S_matrix.h5'
S.to_hdf(fn, key = str(0), mode = 'w')

lp_path = '/data2/hratch/human_me/other/test_lp/'
toy_me_model.pickle(lp_path + 'working_version_' + str(0) + '.pickle')

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '0'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though


In [11]:
# tol = max([abs(i) for i in res['no_dummy']['infeasible_reactions'].values()])
# print('Tolerance: {}'.format(tol))
# fail = {k:v for k,v in res['dummy']['infeasible_reactions'].items() if abs(v) >= tol}

# fail_ids = set(pd.Series(list(fail.keys())).apply(lambda x: x.split('_')[0]))
# fail_metabs = [m for m in res['dummy']['model'].metabolites if m.id.split('_')[0] in fail_ids]